In [17]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/SantanderCS'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
import numpy as np
import pandas as pd
from sklearn.ensemble    import RandomForestClassifier
from sklearn.metrics     import classification_report, roc_auc_score, f1_score, recall_score
from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils    import get_model_train_eval
from utils.feature_engineering import add_statistical_features, drop_highly_correlated_features


In [60]:
# 데이터 로딩 및 기본 전처리
train, test = load_data()
X_features, y_labels = split_features_target(train)
X_test = test.drop(columns=['ID'], axis=1)

In [ ]:
# zero_count_rate 제거했을때 제거될 컬럼수 149개 잔존
X_features, X_test = remove_zero_columns2(X_features, X_test, 0.99)


In [62]:
print(X_features.shape)

# var3 처리
X_features['var3'] = X_features['var3'].replace(-999999, 2)

(76020, 149)


In [ ]:
# # add_statistical_features() before corr
# X_features = add_statistical_features(X_features)
# X_test     = add_statistical_features(X_features)

In [ ]:
# 상관계수 높은 feature들 삭제하기 default 0.95
X_reduced, to_drop = drop_highly_correlated_features(X_features)
X_test_reduced = X_test.drop(to_drop, axis=1)

In [64]:
print(X_reduced.shape, X_test_reduced.shape)

(76020, 96) (75818, 96)


In [ ]:
# 삭제된 컬럼 개수 확인
print("Train에서 삭제된 컬럼 개수:", len(to_drop))

# for i, col in enumerate(sorted(to_drop), start=1):
#     print(f"{i:>2} : {col}")

In [48]:
# 리스트를 Pandas Series로 변환
series = pd.Series(sorted(to_drop), name="Dropped_Columns")

# CSV 파일로 저장
series.to_csv("../data/99perCorr95DroppedColumns_20251122.xls", index=False)


In [ ]:
# 스케일링 
# X_train_scaled, X_test_scaled, scaler = scale_data(X_reduced, X_test_reduced)

In [ ]:
# 학습/테스트 데이터 분리
# X_train, X_val, y_train, y_val = data_split(X_train_scaled, y_labels)
X_train, X_val, y_train, y_val = data_split(X_reduced, y_labels)

In [ ]:
model_name = 'RandomForest_99per_corrTh95_NoTScaled_ne390_maxDepth25_classWeight12_msl1_mss7' 
# model_name = 'RandomForest_99per_corrTh95_Scaled_ne390_maxDepth25_classWeight12_msl1_mss7' 
# Best Option 적용
rf_clf = RandomForestClassifier(
  random_state = 0,
  n_estimators = 390,
  max_depth    = 25, 
  class_weight = {0:1, 1:2}, # 클래스별 가중치
  min_samples_leaf  = 1, 
  min_samples_split = 7,
  n_jobs            = -1 # 병렬처리 여부   
  
)

# 함수 이용
get_model_train_eval(rf_clf, model_name, X_train, X_val, y_train, y_val)

✓ 모델 저장 완료: ../models\RandomForest_99per_corrTh95_Scaled_ne390_maxDepth25_classWeight12_msl1_mss7.pkl
  파일 크기: 79.76 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8409, 정확도: 0.9602, 정밀도: 0.3636, 재현율: 0.0066, F1: 0.0131
오차행렬:
[[14595     7]
 [  598     4]]
실행 시간: 11.42557954788208


In [ ]:
#threshold  dropped_features  AUC       F1        Recall
# 0.95      53                0.842169  0.016287  0.008306 (Not Scaled)
# 0.95      53                0.8409    0.0131    0.0066   (Scaled)
# 0.95      57                0.8359    0.0350    0.0183   (Not Scaled)